# Онлайн-деканат: обучение в Colab
Обучение компактных компонентов и сохранение весов. Данные синтетические, результаты не доказывают качество на реальных документах. Выберите Runtime → Change runtime type → GPU. Не обходите ограничения Colab.

In [ ]:
import torch, platform
print(platform.python_version())
print(torch.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU не подключена")


## Загрузка проекта
Загрузите ZIP с папкой online-dekanat. Архив содержит исходники; скрипт проверяет пути перед распаковкой. Альтернатива: клонировать свой репозиторий обычной командой git clone.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
root = Path("/content/project").resolve()
root.mkdir(exist_ok=True)
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as archive:
            for member in archive.infolist():
                target = (root / member.filename).resolve()
                if not target.is_relative_to(root):
                    raise ValueError("Unsafe archive path")
            archive.extractall(root)
projects = list(root.rglob("train_visual.py"))
assert projects, "В ZIP отсутствуют scripts/train_visual.py"
project = projects[0].parent.parent
import os
os.chdir(project)
print(project)


In [ ]:
%pip install -r requirements.txt


## Генерация и обучение
Размер batch можно уменьшить при нехватке VRAM. Последний checkpoint сохраняется каждую эпоху. Данные train/val/test разделены по семействам шаблонов.

In [ ]:
!python scripts/generate_data.py --count 3000
!python scripts/train_fields.py
!python scripts/train_visual.py --epochs 12 --batch 32
!python scripts/prepare_detection.py --synthetic 600
!python scripts/train_detection.py --epochs 30 --batch 8
!python scripts/train_ocr.py --epochs 8 --count 4000 --lr 0.00003
!python scripts/make_demo.py
!python scripts/make_audit.py
!python scripts/select_ocr.py


## Продолжение прерванного запуска
После восстановления проекта, данных и checkpoint можно выполнить следующую ячейку.

In [ ]:
# !python scripts/train_visual.py --epochs 12 --batch 32 --resume models/visual.last.pt


## Проверка и скачивание
Перенесите models/ в локальный проект. Установите одинаковые версии scikit-learn/joblib для загрузки классификатора полей. Итоговую скорость измеряйте на ПК для демо.

In [ ]:
!python scripts/make_audit.py --split audit_final
!python scripts/evaluate.py --split audit_final --limit 36 --ocr selected
!python scripts/stress_test.py
!python scripts/make_photo_examples.py
!python scripts/stress_camera.py --mode photo --set fresh
!python scripts/validate_detection_real.py
!python scripts/smoke_demo.py
!python scripts/inventory_data.py
!python -m pytest -q
import zipfile
with zipfile.ZipFile("trained_project_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for directory in ["models", "artifacts"]:
        for path in Path(directory).rglob("*"):
            if path.is_file():
                archive.write(path, path)
files.download("trained_project_results.zip")


## Сохранение в Drive (необязательно)
Подключение Drive требует вашего подтверждения в интерфейсе Google. Для длинного обучения сохраняйте checkpoint после каждой эпохи во внешнее хранилище; текущий скрипт сохраняет его локально.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !python scripts/train_visual.py --data data --epochs 12 --output /content/drive/MyDrive/dekanat/visual.pt
